# 12 · Optional Download of Linked Official Documentation

**Run only if you want to fetch the actual documentation from the Cisco, Agilent and Google URLs.** Notebook 09 catalogs these links but does not treat the URLs as technical evidence. This notebook performs HTTPS fetching from allowlisted hosts with bounded downloads, checks redirect targets, and indexes successfully extracted PDF or HTML content. Some sites may block automated retrieval or render content with JavaScript; failures remain `NOT_INDEXED`. Use the existing course sources if network access is unavailable. No claims about product part compatibility are made merely from a link list.

In [ ]:
%run 09_course_document_import.ipynb
import httpx, ipaddress, socket, html
from html.parser import HTMLParser
from urllib.parse import urljoin, urlsplit

# To download official source material, explicitly switch to True. Network required.
DOWNLOAD_OFFICIAL_SOURCES = False
MAX_SOURCE_FETCHES = 8
TRUSTED_SUFFIXES=('cisco.com','agilent.com','cloud.google.com','docs.cloud.google.com','ibm.com')

class PlainPage(HTMLParser):
    def __init__(self):super().__init__();self.skip=0;self.parts=[]
    def handle_starttag(self,tag,attrs):
        if tag in ('script','style','noscript'):self.skip+=1
        elif tag in ('p','h1','h2','h3','li','td','th','title'):self.parts.append('\n')
    def handle_endtag(self,tag):
        if tag in ('script','style','noscript') and self.skip:self.skip-=1
        elif tag in ('p','h1','h2','h3','li','td','th','title'):self.parts.append('\n')
    def handle_data(self,data):
        if not self.skip and data.strip():self.parts.append(html.unescape(data.strip())+' ')

def permitted_url(url):
    p=urlsplit(url)
    host=(p.hostname or '').lower()
    if p.scheme!='https' or p.username or p.password or p.port not in (None,443):return False
    if host not in TRUSTED_SUFFIXES and not any(host.endswith('.'+base) for base in TRUSTED_SUFFIXES):return False
    # Resolve and reject private/link-local destinations even for allowlisted names.
    try:
        addrs=socket.getaddrinfo(host,443,type=socket.SOCK_STREAM)
        if not addrs:return False
        return all(ipaddress.ip_address(info[4][0]).is_global for info in addrs)
    except OSError:return False

def fetch_official(url,client,max_bytes=20*1024*1024):
    current=url
    for _ in range(4):
        if not permitted_url(current):raise ValueError('URL was not approved for external fetching')
        with client.stream('GET',current,follow_redirects=False) as resp:
            if resp.status_code in (301,302,303,307,308):
                loc=resp.headers.get('location')
                if not loc:raise ValueError('Redirect without location')
                current=urljoin(current,loc)
                continue
            resp.raise_for_status()
            mime=resp.headers.get('content-type','').lower()
            data=bytearray()
            for chunk in resp.iter_bytes():
                data.extend(chunk)
                if len(data)>max_bytes:raise ValueError('Source larger than 20 MiB')
            return bytes(data),mime,current
    raise ValueError('Too many redirects')

def download_and_ingest_catalog(max_sources=MAX_SOURCE_FETCHES):
    results=[]
    with httpx.Client(timeout=httpx.Timeout(20,connect=10),headers={'User-Agent':'CourseResearchPrototype/1.0'}) as client:
        for item in catalog[:max_sources]:
            try:
                data,mime,url=fetch_official(item['url'],client)
                is_pdf=data.startswith(b'%PDF-')
                if is_pdf:
                    filename=(Path(urlsplit(url).path).name or 'source.pdf')[:160]
                    if not filename.lower().endswith('.pdf'): filename+='.pdf'
                    content=data
                elif 'html' in mime:
                    parser=PlainPage();parser.feed(data.decode('utf-8',errors='replace'))
                    content=' '.join(''.join(parser.parts).split()).encode('utf-8')
                    filename='official-'+hashlib.sha256(url.encode()).hexdigest()[:16]+'.txt'
                    if len(content)<120:raise ValueError('Web page text too short; may require Javascript or authentication')
                else:raise ValueError('Unsupported content type')
                status=ingest_blob(PROJECT_ID,item['domain'],filename,content,
                                   source_type='official-web-downloaded',source_url=url)
                results.append({'url':url,'domain':item['domain'],'status':'INDEXED',**status})
            except Exception as exc:
                results.append({'url':item['url'],'status':'NOT_INDEXED','reason':str(exc)[:220]})
    output=COURSE/'official_fetch_results.json';output.write_text(json.dumps(results,indent=2))
    return results

if DOWNLOAD_OFFICIAL_SOURCES:
    downloaded=download_and_ingest_catalog()
    print(json.dumps(downloaded,indent=2)[:8000])
else:
    print('Live downloads disabled. Set DOWNLOAD_OFFICIAL_SOURCES=True to fetch authorized official PDFs/pages over HTTPS.')
